# Lab 10: Applying Functions and Iteration

Please complete this notebook by filling in the cells provided. Before you begin, execute the previous cell to load the provided tests.

**Helpful Resource:**
- [Python Reference](http://data8.org/sp24/reference/): Cheat sheet of helpful array & table methods!

**Recommended Readings**:

* [Tabular Thinking Guide](https://drive.google.com/file/d/1D9PXUgDLXjFBXZoyGVSKpzk0nKQziDTP/view?usp=sharing)
* [Applying Functions](https://www.inferentialthinking.com/chapters/08/1/Applying_a_Function_to_a_Column.html)
* [Conditionals](https://www.inferentialthinking.com/chapters/09/1/Conditional_Statements.html)
* [Iteration](https://www.inferentialthinking.com/chapters/09/2/Iteration.html)

Please complete this notebook by filling in the cells provided. **Before you begin, execute the cell below to setup the notebook by importing some helpful libraries.** Each time you start your server, you will need to execute this cell again.

## 1. 2023 Cal Football Season

In [ ]:
!apt-get install texlive texlive-xetex texlive-latex-extra pandoc
!pip install pypandoc
!pip install datascience

In [ ]:
# Connect Google Drive to Colab so you can access your files
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/')

In [ ]:
# Run this cell to set up the notebook, but please don't change it.

# These lines import the Numpy and Datascience modules.
import numpy as np
from datascience import *

# These lines do some fancy plotting magic.
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

Jonathan and Jadyn are trying to analyze how well the Cal football team performed in the 2023 season. A football game is divided into four periods, called quarters. The number of points Cal scored in each quarter and the number of points their opponent scored in each quarter are stored in a table called `cal_fb.csv`.

In [ ]:
# Just run this cell
# Read in the cal_fb csv file
games = Table().read_table("./DS/cal_fb.csv")
games.show()

Let's start by finding the total points each team scored in a game.

**Question 1.** Write a function called `sum_scores`.  It should take four arguments, where each argument represents integers corresponding to the team's score for each quarter. It should return the team's total score for that game. **(2 Points)**

*Hint:* Don't overthink this question!



In [ ]:
def sum_scores(q1, q2, q3, q4):
    '''Returns the total score calculated by adding up the score of each quarter'''
    return q1 + q2 + q3 + q4

sum_scores(14, 7, 3, 0) #DO NOT CHANGE THIS LINE

**Question 2.** Create a new table `final_scores` with three columns in this *specific* order: `Opponent`, `Cal Score`, `Opponent Score`. You will have to create the `Cal Score` and `Opponent Score` columns. Use the function `sum_scores` you just defined in the previous question for this problem. **(5 Points)**

*Hint:* If you want to apply a function that takes in multiple arguments, you can pass multiple column names as arguments in `tbl.apply()`. The column values will be passed into the corresponding arguments of the function. Take a look at the Python Reference Sheet and Lecture 13's demo for syntax.

*Note:* If you’re running into issues creating `final_scores`, check that `cal_scores` and `opp_scores` output what you want. If you're encountering `TypeError`s, check the [Python Reference](https://www.data8.org/sp24/reference/) to see if the inputs/outputs of the function are what you expect.


In [ ]:
cal_scores = games.apply(sum_scores, "Cal 1Q", "Cal 2Q", "Cal 3Q", "Cal 4Q")
opp_scores = games.apply(sum_scores, "Opp 1Q", "Opp 2Q", "Opp 3Q", "Opp 4Q")
final_scores = Table().with_columns(
    "Opponent", games.column("Opponent"),
    "Cal Score", cal_scores,
    "Opponent Score", opp_scores
)
final_scores

We can get specific row objects from a table. You can use `tbl.row(n)` to get the `n`th row of a table. `row.item("column_name")` will allow you to select the element that corresponds to `column_name` in a particular row. Here's an example:

In [ ]:
# Just run this cell
# We got the Axe!
games.row(9) # <-- this will return a row object

In [ ]:
# Just run this cell
games.row(9).item("Cal 4Q") # <-- this will return a item (e.g. an int) from a row object

**Question 3.** We want to see for a particular game whether or not Cal lost. Write a function called `did_cal_lose`.  It should take one argument: a **row object** from the `final_scores` table. It should return either `True` if Cal's score was less than the Opponent's score, and `False` otherwise. **(5 Points)**

*Note 1*: "Row object" means a row from the table extracted (behind the scenes) using `tbl.row(index)` that contains all the data for that specific row. It is **not** the index of a row. Do not try and call `final_scores.row(row)` inside of the function.

*Note 2*: If you're still confused by row objects, try printing out `final_scores.row(1)` in a new cell to visually see what it looks like! This piece of code is pulling out the row object located at index 1 of the `final_scores` table and returning it. When you display it in a cell, you'll see that it is not located within a table, but is instead a standalone row object!

Hint: You can use comparators like <, >, <=, >=, ==, and != to compare items from a row object


In [ ]:
def did_cal_lose(row):
    return row.item("Cal Score") < row.item("Opponent Score")

did_cal_lose(final_scores.row(1)) #DO NOT CHANGE THIS LINE

**Question 4.** Jonathan and Jadyn wants to see how Cal did against every opponent during the 2023 season. Using the `final_scores` table:

1. Assign `results_array` to an array of `True` and `False` values that correspond to whether or not Cal lost.
2. Add `results_array` to the `final_scores` table in a column named `Results`, and assign this to `final_scores_with_results`.
3. Then, respectively assign the number of wins and losses Cal had to `cal_wins` and `cal_losses`.

**(10 Points)**

*Hint*: `tbl.apply()` might be helpful. Refer to the [Python Reference](https://www.data8.org/sp24/reference/) if you're unsure how it works!


In [ ]:
results_array = final_scores.apply(did_cal_lose)
final_scores_with_results = final_scores.with_column("Results", results_array)
cal_losses = results_array.sum()
cal_wins = len(results_array) - cal_losses

# Don't delete or edit the following line:
print(f"In the 2023 Season, Cal Football won {cal_wins} games and lost {cal_losses} games. Go Bears! 🐻")

## 2. Unrolling Loops


"Unrolling" a `for` loop means to manually write out all the code that it executes.  The result is code that does the same thing as the loop, but without the structure of the loop.  For example, for the following loop:

    for num in np.arange(3):
        print("The number is", num)

The unrolled version would look like this:

    print("The number is", 0)
    print("The number is", 1)
    print("The number is", 2)


Unrolling a `for` loop is a great way to understand what the loop is doing during each step. In this exercise, you'll practice unrolling a `for` loop.


In the question below, write code that does the same thing as the given code, but with any `for` loops unrolled.  It's a good idea to run both your answer and the original code to verify that they do the same thing.  (Of course, if the code does something random, you'll get a different random outcome than the original code!)

**Question 1.** Unroll the code below.


In [ ]:
for joke_iteration in np.arange(3):
    print("Knock. " * (joke_iteration + 1))
    print("Who's there?")
    print("Banana.")
    print("Banana who?")
print("Knock, knock.")
print("Who's there?")
print("Orange.")
print("Orange who?")
print("Orange you glad I didn't say banana?")

In [ ]:
print("Knock. ")
print("Who's there?")
print("Banana.")
print("Banana who?")
print("Knock. Knock. ")
print("Who's there?")
print("Banana.")
print("Banana who?")
print("Knock. Knock. Knock. ")
print("Who's there?")
print("Banana.")
print("Banana who?")
print("Knock, knock.")
print("Who's there?")
print("Orange.")
print("Orange who?")
print("Orange you glad I didn't say banana?")

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a pdf file for you to submit. **Please save before exporting!**

In [ ]:
# should change the directory and file name matching to yours
!jupyter nbconvert './DS/lab10(SSU).ipynb' --to pdf